# 01 — The frozen disc localiser, and the crops every label source will share

**Runtime → Change runtime type → GPU → L4.** An A100 buys nothing here: this is one short
training run and then a lot of disk writing.

Audit fix A1. If the crop moves when the convention changes, a model trained on a liberal
expert is also looking at a different field of view, and nothing downstream can separate the
two. So one localiser is trained **once**, on the official merge only, then frozen; it emits
one box per eye; and all nine label sources are cropped with that same box.

Containment of all seven experts' discs is then **proven on every eye**, not assumed.

About 30–45 minutes. It resumes if Colab drops you.


In [ ]:
# --- environment -----------------------------------------------------------
# WHERE THINGS LIVE, and why:
#   PROJECT  this project's code (src/, analysis/)   -> Drive, or a GitHub clone
#   RAW      the 1.5 GB dataset                      -> LOCAL disk: Drive I/O is slow,
#                                                       and re-fetching costs ~3 minutes
#   WORK     crops, weights, results                 -> Drive, so a disconnect costs nothing
REPO_URL = "https://github.com/Aduitya9/Opiotica.git"   # cloned fresh each session
DRIVE_PROJECT = "/content/drive/MyDrive/whose-eyes"     # offline fallback if REPO_URL is ""
WORK = "/content/drive/MyDrive/whose-eyes-work"
RAW = "/content/raw"

import json, os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")

if REPO_URL:
    if not os.path.isdir("/content/project"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/project"], check=True)
    PROJECT = "/content/project"
else:
    PROJECT = DRIVE_PROJECT
assert os.path.isdir(os.path.join(PROJECT, "src")), (
    f"No src/ under {PROJECT}. Copy the `whose-eyes` folder into your Drive at "
    f"{DRIVE_PROJECT}, or set REPO_URL above.")
sys.path.insert(0, PROJECT)
for sub in ("crops", "runs", "results"):
    os.makedirs(os.path.join(WORK, sub), exist_ok=True)
os.makedirs(RAW, exist_ok=True)

print(f"project : {PROJECT}")
print(f"work    : {WORK}   (Drive - persists between sessions)")
print(f"raw     : {RAW}   (LOCAL - wiped with this VM, re-fetched on demand)")


In [ ]:
!pip -q install "segmentation-models-pytorch==0.4.0" "timm>=1.0.9" || pip -q install segmentation-models-pytorch

import hashlib, time
import numpy as np, pandas as pd, torch
from PIL import Image
import segmentation_models_pytorch as smp

from src import refuge, crops as crops_mod, labels as labels_mod, train as train_mod


In [ ]:
# --- preflight: prove this session can finish, before spending L4 time --------
from src import colab

GPU, VRAM = colab.require_gpu()
BATCH = colab.batch_size_for(VRAM)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

colab.require({
    os.path.join(WORK, "results", "inventory.csv"): "notebook 00",
    os.path.join(WORK, "results", "png_measurements.csv"): "notebook 00",
    os.path.join(WORK, "results", "crop_margin.json"): "notebook 00",
})
RELEASE = colab.ensure_release(RAW)          # fetches it if this VM does not have it
MARGIN = json.load(open(os.path.join(WORK, "results", "crop_margin.json")))["margin"]

inventory = pd.read_csv(os.path.join(WORK, "results", "inventory.csv"), dtype={"image_id": str})
png = pd.read_csv(os.path.join(WORK, "results", "png_measurements.csv"), dtype={"image_id": str})
folder_of = {i: f for _, i, f in refuge.iter_images(RELEASE)}

print(f"\nready: {GPU} {VRAM:.0f} GB, batch {BATCH}, {len(inventory)} eyes, margin {MARGIN}")


In [ ]:
# --- the fixed split, drawn once and saved -----------------------------------
# 320 Zeiss eyes to train on, 80 held back for early stopping, 800 Canon eyes to evaluate on.
# Byte-identical for every label source, for the whole study.
SPLIT_FILE = os.path.join(WORK, "results", "zeiss_train80_val20.json")
if os.path.exists(SPLIT_FILE):
    split_ids = json.load(open(SPLIT_FILE))
    print("reusing the existing split - it is never redrawn")
else:
    zeiss = inventory[inventory.camera == "Zeiss"].image_id.tolist()
    shuffled = list(np.random.default_rng(20261113).permutation(zeiss))
    split_ids = {"train": sorted(shuffled[:320]), "val": sorted(shuffled[320:]),
                 "eval": sorted(inventory[inventory.camera == "Canon"].image_id)}
    json.dump(split_ids, open(SPLIT_FILE, "w"), indent=2)
    print("split drawn and frozen")
print({k: len(v) for k, v in split_ids.items()})


In [ ]:
# --- localiser dataset: whole image -> official disc only --------------------
LOC_SIZE = 512

class LocaliserSet(torch.utils.data.Dataset):
    def __init__(self, ids, augment=False):
        self.ids, self.augment = list(ids), augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, k):
        image_id = self.ids[k]
        folder = folder_of[image_id]
        jpg = next(f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg")))
        image = Image.open(os.path.join(folder, jpg)).convert("RGB").resize((LOC_SIZE, LOC_SIZE))
        _, disc = refuge.official_masks(folder, image_id)
        disc = np.array(Image.fromarray(disc.astype(np.uint8) * 255)
                        .resize((LOC_SIZE, LOC_SIZE), Image.NEAREST)) > 127
        x = np.asarray(image, dtype=np.float32).transpose(2, 0, 1) / 255.0
        y = disc[None].astype(np.float32)
        if self.augment and np.random.rand() < 0.5:
            x, y = x[:, :, ::-1].copy(), y[:, :, ::-1].copy()
        return torch.from_numpy(x), torch.from_numpy(y)

print(len(LocaliserSet(split_ids["train"])), "training eyes for the localiser")


In [ ]:
# --- train the localiser ONCE, then freeze it --------------------------------
LOC_DIR = os.path.join(WORK, "runs", "localiser")
os.makedirs(LOC_DIR, exist_ok=True)
WEIGHTS = os.path.join(LOC_DIR, "best.pt")

localiser = smp.Unet("efficientnet-b4",
                     encoder_weights=None if os.path.exists(WEIGHTS) else "imagenet",
                     in_channels=3, classes=1)

if os.path.exists(WEIGHTS):
    localiser.load_state_dict(torch.load(WEIGHTS, weights_only=True))
    localiser = localiser.cuda().eval()
    print("localiser already trained - loading the frozen weights, not retraining")
else:
    train_mod.set_seed(20261113)
    localiser = localiser.cuda()
    opt = torch.optim.AdamW(localiser.parameters(), lr=3e-4, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda")
    loaders = {
        "train": torch.utils.data.DataLoader(LocaliserSet(split_ids["train"], augment=True),
                                             batch_size=BATCH, shuffle=True, num_workers=2,
                                             drop_last=True, pin_memory=True),
        "val": torch.utils.data.DataLoader(LocaliserSet(split_ids["val"]), batch_size=BATCH,
                                           num_workers=2, pin_memory=True),
    }
    best = float("inf")
    for epoch in range(30):
        line = {}
        for phase in ("train", "val"):
            localiser.train(phase == "train")
            total = count = 0
            with torch.set_grad_enabled(phase == "train"):
                for x, y in loaders[phase]:
                    x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
                    with torch.amp.autocast("cuda"):
                        loss = train_mod.dice_bce(localiser(x), y)
                    if phase == "train":
                        opt.zero_grad(set_to_none=True)
                        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                    total += loss.item() * x.size(0); count += x.size(0)
            line[phase] = total / count
        marker = ""
        if line["val"] < best:
            best, marker = line["val"], "  *"
            torch.save(localiser.state_dict(), WEIGHTS)
        print(f"epoch {epoch:2d}  train {line['train']:.4f}  val {line['val']:.4f}{marker}")
    localiser.load_state_dict(torch.load(WEIGHTS, weights_only=True))
    localiser.eval()

for p in localiser.parameters():
    p.requires_grad = False
print("\nFROZEN. This model is never trained again in this project.")


In [ ]:
# --- one box per eye, from the frozen localiser ------------------------------
BOX_FILE = os.path.join(WORK, "crops", "boxes.csv")
SHA_FILE = os.path.join(WORK, "crops", "boxes.sha256.json")

# Bumped whenever the way a box is derived changes. Boxes written under an older rule are
# rebuilt automatically - "frozen" must never mean "keep the ones with the known bug in them".
#   v1  bounding box of every predicted pixel  (a stray pixel ballooned 68 Canon boxes)
#   v2  largest connected component first
BOX_RULE = "v2-largest-component"

# Frozen means frozen: boxes built under the CURRENT rule are reused, never recomputed. AMP
# inference is not bit-exact, so a second pass could move a box by a pixel and change the
# sha256 of a file whose whole purpose is to be fixed.
stale = os.path.exists(SHA_FILE) and json.load(open(SHA_FILE)).get("box_rule") != BOX_RULE
if stale:
    print(f"boxes were built under an older rule - rebuilding under {BOX_RULE}\n")
if os.path.exists(BOX_FILE) and not stale:
    boxes = pd.read_csv(BOX_FILE, dtype={"image_id": str})
    meta = json.load(open(SHA_FILE))
    failures = meta.get("failures", [])
    print(f"reusing {len(boxes)} frozen boxes   sha256 {meta['sha256'][:16]}...   "
          f"margin {meta['margin']}   (delete boxes.csv to rebuild)")
else:
    rows, failures = [], []
    all_ids = inventory.image_id.tolist()
    with torch.no_grad():
        for n, image_id in enumerate(all_ids, 1):
            folder = folder_of[image_id]
            jpg = next(f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg")))
            full = Image.open(os.path.join(folder, jpg)).convert("RGB")
            W_full, H_full = full.size
            x = torch.from_numpy(np.asarray(full.resize((LOC_SIZE, LOC_SIZE)), dtype=np.float32)
                                 .transpose(2, 0, 1)[None] / 255.0).cuda()
            with torch.amp.autocast("cuda"):
                pred = torch.sigmoid(localiser(x))[0, 0].float().cpu().numpy() > 0.5
            if not pred.any():
                failures.append({"image_id": image_id, "reason": "localiser found no disc"})
                continue
            # The disc is ONE blob. Without this, a stray false-positive pixel in a corner
            # balloons the bounding box: it put 68 Canon eyes in boxes up to 5.3x too large.
            pred = crops_mod.largest_component(pred)
            top, left, bottom, right = crops_mod.mask_box(pred)
            sy, sx = H_full / LOC_SIZE, W_full / LOC_SIZE
            disc_full = np.zeros((H_full, W_full), dtype=bool)
            disc_full[int(top * sy):int(bottom * sy), int(left * sx):int(right * sx)] = True
            box = crops_mod.square_box(disc_full, (H_full, W_full), margin=MARGIN)
            rows.append({"image_id": image_id, "height": H_full, "width": W_full,
                         "top": box[0], "left": box[1], "bottom": box[2], "right": box[3]})
            if n % 200 == 0:
                print(f"  {n}/{len(all_ids)}")

    boxes = pd.DataFrame(rows)
    boxes.to_csv(BOX_FILE, index=False)
    digest = hashlib.sha256(open(BOX_FILE, "rb").read()).hexdigest()
    json.dump({"sha256": digest, "margin": MARGIN, "n": len(boxes), "gpu": GPU,
               "box_rule": BOX_RULE, "failures": failures}, open(SHA_FILE, "w"), indent=2)
    print(f"\n{len(boxes)} boxes frozen.  sha256 {digest[:16]}...")

print("failures:", failures if failures else "none")


In [ ]:
# --- PROVE the box holds every expert's disc ---------------------------------
def column(frame, name):
    return np.array([[int(v) for v in s.split(",")] for s in frame[name]], dtype=int)

check = boxes.merge(png, on="image_id", how="inner")
expert = np.stack([column(check, f"box_user{k}") for k in range(1, 8)])
union = np.stack([expert[:, :, 0].min(0), expert[:, :, 1].min(0),
                  expert[:, :, 2].max(0), expert[:, :, 3].max(0)], axis=1)
report = pd.DataFrame({
    "image_id": check.image_id,
    "slack_top": union[:, 0] - check.top.to_numpy(),
    "slack_left": union[:, 1] - check.left.to_numpy(),
    "slack_bottom": check.bottom.to_numpy() - union[:, 2],
    "slack_right": check.right.to_numpy() - union[:, 3],
})
report["contains_union"] = (report[["slack_top", "slack_left",
                                    "slack_bottom", "slack_right"]] >= 0).all(axis=1)
report["padding_fraction"] = [
    crops_mod.padding_fraction((r.top, r.left, r.bottom, r.right), (r.height, r.width))
    for r in check.itertuples()]
report.to_csv(os.path.join(WORK, "results", "containment.csv"), index=False)

padded = report[report.padding_fraction > 0]
print(f"eyes whose box extends past the image border: {len(padded)} / {len(report)}"
      f"  (max padding {100 * report.padding_fraction.max():.1f}% of the crop)")
print("   that padding is black, which is what a fundus surround already is\n")

# Amendment 4: padding is far commoner on Zeiss, which is the split we train on. It cancels
# in Delta_model (same box for every label source) but the asymmetry is reported, and
# notebook 02 repeats the gate on the unpadded evaluation eyes as a sensitivity check.
split_pad = (report.merge(inventory[["image_id", "split", "camera"]], on="image_id")
             .assign(padded=lambda d: d.padding_fraction > 0)
             .groupby(["split", "camera"]).padded.agg(["sum", "count"]))
split_pad["percent"] = (100 * split_pad["sum"] / split_pad["count"]).round(1)
print("padding by split - expect roughly 33% on Zeiss against 2-4% on Canon:")
print(split_pad.to_string())
print()
# How far is the frozen localiser from the official-merge disc it was trained on? The box has
# a median 96 px of slack at margin 1.8, so a few pixels are harmless - but the localiser
# trains on Zeiss and is applied to Canon, so any generalisation failure shows up HERE, as
# centre error concentrated on one camera, before it can quietly cost evaluation eyes.
official = np.array([[int(v) for v in s.split(",")] for s in check.box_official])
err = pd.DataFrame({
    "split": check.split.to_numpy(),          # png_measurements already carries the split
    "centre_dy": (official[:, 0] + official[:, 2]) / 2
                 - (check.top.to_numpy() + check.bottom.to_numpy()) / 2,
    "centre_dx": (official[:, 1] + official[:, 3]) / 2
                 - (check.left.to_numpy() + check.right.to_numpy()) / 2,
})
err["centre_error_px"] = np.hypot(err.centre_dy, err.centre_dx)
print("localiser box centre error vs the official-merge disc, by split (px):")
print(err.groupby("split").centre_error_px.describe()[["mean", "50%", "max"]].round(1).to_string())
print("   a camera-specific jump here means the localiser did not generalise Zeiss -> Canon\n")

# Containment only catches boxes that are too SMALL or mis-centred away from the disc. A box
# five times too large contains everything and passes - while shrinking the disc to a fifth of
# its proper scale. That happened for real, so the opposite direction is checked too.
implied = (check.bottom.to_numpy() - check.top.to_numpy()) / MARGIN
official_diam = np.maximum(official[:, 2] - official[:, 0], official[:, 3] - official[:, 1])
size_ratio = implied / official_diam
print("box size vs the official-merge disc (1.00 = the localiser agrees), by split:")
print(pd.DataFrame({"split": check.split.to_numpy(), "ratio": size_ratio})
      .groupby("split").ratio.describe()[["mean", "50%", "max"]].round(2).to_string())
oversized = int(((size_ratio > 1.25) | (size_ratio < 0.8)).sum())
print(f"\nimplausible boxes (>25% too big or >20% too small): {oversized} / {len(check)}")
if oversized > 0.02 * len(check):
    raise SystemExit(
        f"\nSTOP. {oversized} boxes are the wrong size for their disc. Containment will not "
        "catch this - an over-large box contains everything while putting the disc at a "
        "fraction of its scale.\nCheck the localiser before exporting crops; do not train "
        "on these.")

bad = report[~report.contains_union]
print(f"\neyes where the frozen box clips an expert's disc: {len(bad)} / {len(report)}")
if len(bad):
    print("   by split - failures concentrated on Canon would be a localiser problem, "
          "not a margin problem:")
    print("   " + bad.merge(inventory[["image_id", "split"]], on="image_id")
          .groupby("split").size().to_string().replace("\n", "\n   "))
print("\nsmallest slack in pixels (how close the box came to clipping):")
print(report[["slack_top", "slack_left", "slack_bottom", "slack_right"]].min().to_string())
if len(bad):
    display(bad.head(20))
    print("\nThese eyes are EXCLUDED and the count goes in the paper. They are not dropped "
          "quietly, and the margin is NOT raised now - that would be tuning after the fact.")


In [ ]:
# --- export the crops: one .npz per eye, 19 arrays, identical pixels ---------
CROP = train_mod.CONFIG.crop_size
keep = set(report.loc[report.contains_union, "image_id"])
train_set, val_set = set(split_ids["train"]), set(split_ids["val"])
manifest, t0 = [], time.time()

# A crop is only reusable if it was cut with the boxes we are holding now. Without this
# stamp, fixing a box bug would leave the old crops in place and the fix would do nothing.
STAMP = os.path.join(WORK, "crops", "built_from.json")
box_sha = json.load(open(SHA_FILE))["sha256"]
previous = json.load(open(STAMP))["sha256"] if os.path.exists(STAMP) else None
reuse = previous == box_sha
if previous and not reuse:
    print("the boxes changed since these crops were written - re-cutting every eye\n")

for n, (_, row) in enumerate(boxes.iterrows(), 1):
    image_id = row.image_id
    rel = os.path.join("crops", f"{image_id}.npz")
    if image_id not in keep:
        continue
    if reuse and os.path.exists(os.path.join(WORK, rel)):   # resume after a disconnect
        manifest.append({"image_id": image_id, "npz": rel,
                         "camera": inventory.loc[inventory.image_id == image_id,
                                                 "camera"].iloc[0],
                         "part": "train" if image_id in train_set
                                 else "val" if image_id in val_set else "eval"})
        continue
    folder = folder_of[image_id]
    jpg = next(f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg")))
    box = (int(row.top), int(row.left), int(row.bottom), int(row.right))

    def cut(arr, nearest=False):
        # crops_mod.crop pads where the box leaves the image. Plain slicing must NOT be used:
        # a negative index wraps around and pastes the opposite side of the retina into the crop.
        piece = crops_mod.crop(arr.astype(np.uint8) * 255 if nearest else arr, box)
        return np.array(Image.fromarray(piece).resize(
            (CROP, CROP), Image.NEAREST if nearest else Image.BILINEAR))

    arrays = {"image": cut(np.asarray(Image.open(os.path.join(folder, jpg)).convert("RGB")))
                       .astype(np.uint8)}
    cups, discs = [], []
    for k in range(1, 8):
        cup, disc = refuge.expert_masks(folder, image_id, k)
        cup, disc = cut(cup, nearest=True) > 127, cut(disc, nearest=True) > 127
        arrays[f"cup_user{k}"], arrays[f"disc_user{k}"] = cup, disc
        cups.append(cup); discs.append(disc)
    ocup, odisc = refuge.official_masks(folder, image_id)
    arrays["cup_official"] = cut(ocup, nearest=True) > 127
    arrays["disc_official"] = cut(odisc, nearest=True) > 127
    arrays["cup_majority"] = labels_mod.fuse(cups, "majority")
    arrays["disc_majority"] = labels_mod.fuse(discs, "majority")

    np.savez_compressed(os.path.join(WORK, rel), **arrays)
    manifest.append({"image_id": image_id, "npz": rel,
                     "camera": inventory.loc[inventory.image_id == image_id, "camera"].iloc[0],
                     "part": "train" if image_id in train_set
                             else "val" if image_id in val_set else "eval"})
    if n % 100 == 0:
        print(f"  {n}/{len(boxes)}   {time.time() - t0:5.0f}s")

manifest = pd.DataFrame(manifest)
manifest.to_csv(os.path.join(WORK, "crops", "manifest.csv"), index=False)
json.dump({"sha256": box_sha, "box_rule": BOX_RULE, "n": len(manifest)},
          open(STAMP, "w"), indent=2)
print(f"\n{len(manifest)} crops ready in {time.time() - t0:.0f}s")
print(manifest.groupby(["part", "camera"]).size().to_string())


In [ ]:
# --- the check that must pass before notebook 02 -----------------------------
# Same eye, two conventions. The IMAGE must be identical and the CUPS must not be.
sample = np.load(os.path.join(WORK, manifest.npz.iloc[0]))
liberal, conservative = sample["cup_user1"].sum(), sample["cup_user7"].sum()
print(f"image array      : {sample['image'].shape} {sample['image'].dtype}")
print(f"cup_user1 pixels : {liberal}")
print(f"cup_user7 pixels : {conservative}")
print(f"ratio            : {liberal / max(conservative, 1):.2f}x")
assert liberal != conservative, (
    "user1 and user7 produced the same cup - the export is wrong and nothing after this "
    "point means anything.")
print("\nOK: one image, different outlines. That difference is the whole experiment.")


In [ ]:
# --- Hugging Face: the archival copy ------------------------------------------
# The token comes from Colab Secrets (key icon in the left sidebar) under the name HF_TOKEN.
# It is never written into a cell, a file, or this repository.
#
# NOT `-U`. Colab ships a huggingface_hub that already does everything used here, and
# force-upgrading it breaks the preinstalled `datasets`, which pins huggingface-hub<2.0:
#     datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have 2.4.0
# Nothing needed the newer version, so nothing should have asked for it.
!pip -q install huggingface_hub

from huggingface_hub import HfApi

HF_REPO = "Aduitya9/opiotica-whose-eyes"          # PRIVATE: it holds REFUGE-derived pixels
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as error:
    HF_TOKEN = None
    print("No HF_TOKEN in Colab Secrets:", error)

PUSH_TO_HF = HF_TOKEN is not None
hf = HfApi(token=HF_TOKEN) if PUSH_TO_HF else None
print("upload enabled" if PUSH_TO_HF else
      "upload DISABLED - add HF_TOKEN under Secrets to archive this run; "
      "everything still works locally and in Drive")


In [ ]:
# --- archive the frozen crops and the decisions that produced them -----------
# Why upload at all: the crops are the one expensive artefact (~45 min to rebuild) and they
# are what makes every later run comparable. Why PRIVATE: they contain REFUGE pixels.
if PUSH_TO_HF:
    for name in ("boxes.csv", "boxes.sha256.json", "manifest.csv"):
        hf.upload_file(path_or_fileobj=os.path.join(WORK, "crops", name),
                       path_in_repo=f"crops/{name}", repo_id=HF_REPO, repo_type="dataset",
                       commit_message=f"Frozen crop metadata: {name}")
    for name in ("inventory.csv", "png_measurements.csv", "crop_margin.json",
                 "containment.csv", "zeiss_train80_val20.json"):
        path = os.path.join(WORK, "results", name)
        if os.path.exists(path):
            hf.upload_file(path_or_fileobj=path, path_in_repo=f"results/{name}",
                           repo_id=HF_REPO, repo_type="dataset",
                           commit_message=f"Notebook 00/01 output: {name}")
    print("metadata uploaded\n")

    # The crops go up as ONE tar, not 1,200 files. A partial upload of 1,200 small LFS files
    # leaves a repo that looks populated but is not; a single file either lands or does not.
    # Nothing downstream needs this - notebook 02 reads the .npz files from Drive - so set
    # UPLOAD_CROPS = False if you would rather not wait.
    UPLOAD_CROPS = True
    if UPLOAD_CROPS:
        tar_path, n_crops, n_bytes = colab.archive_crops(WORK)
        hf.upload_file(path_or_fileobj=tar_path, path_in_repo="crops/crops.tar",
                       repo_id=HF_REPO, repo_type="dataset",
                       commit_message=f"Frozen crops: {n_crops} eyes, image + 18 masks each")
        colab.verify_upload(hf, HF_REPO, {"crops/crops.tar": n_bytes})
        print("   restore with: tar -xf crops.tar -C <your crops folder>")
    print(f"\nhttps://huggingface.co/datasets/{HF_REPO}")


## Done

`whose-eyes-work/crops/` now holds one `.npz` per eye: the image once, and eighteen masks —
seven experts × (cup, disc), plus majority and the official merge.

Next: `02_noise_floor_gate.ipynb`. Stay on the **same GPU type** you used here if you can;
notebook 02 refuses to report a gate computed across mixed hardware.
